Import necessary libraries

In [1]:
# import necessary libraries
import re
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

/mnt/e/TanishProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load environment variables
load_dotenv()

True

Load the document

In [3]:
# load the required document
file_path = "NexaCorp_Enterprise_Policy_Handbook_v5.2.docx"
try:
    loader = UnstructuredWordDocumentLoader(
        file_path,
        mode="elements"                                 # breaks document into structured chunks based on layout
    )
    docs = loader.load()
except Exception as e:
    raise RuntimeError(f"Failed to load document: {file_path}") from e

In [5]:
# types of elements in document
from collections import Counter
categories = Counter(doc.metadata.get("category", "Unknown") for doc in docs)
print(categories)

Counter({'Title': 150, 'NarrativeText': 125, 'ListItem': 113, 'UncategorizedText': 64, 'Table': 43, 'PageBreak': 33, 'Header': 1, 'Footer': 1})


Data Cleaning

In [6]:
# use regular expressions to remove header/footer noise
def is_header_footer(text):
    NOISE_PATTERNS = [
        r"Version\s+\d+[\.\d]*\s*\|",
        r"Internal Restricted",
        r"Not for External Distribution",
        r"©\s*20\d{2}",
        r"CONFIDENTIAL",
        r"Policy.*Handbook"
    ]
    # checks if patterns exist anywhere in the text and returns True if atleast one pattern matches
    return any(re.search(p, text, re.IGNORECASE) for p in NOISE_PATTERNS)

# uses regular expression to check if line belongs to TOC or not
def is_toc_line(text):
    text = text.strip()
    return bool(re.search(
        # check for Part I/3.2+Company Overview & Mission+5
        r"^(Part\s+[IVXLC]+|[\d]+\.[\d]+)\s+.*\s+\d+$",
        text,
        re.IGNORECASE
    ))

# clean the documents by removing unwanted data
def clean_documents(docs):
    cleaned_docs = []
    skip_toc = False
    removed = 0

    for doc in docs:
        text = doc.page_content.strip()
        lower = text.lower()
        category = doc.metadata.get("category", "")

        # Normalize
        text = re.sub(r"\s+", " ", text)

        # Remove header/footer via metadata
        if category in ["Header", "Footer"]:
            removed += 1
            continue

        # Regex-based removal
        if is_header_footer(text):
            removed += 1
            continue

        # Detect TOC start
        if "table of contents" in lower:
            skip_toc = True
            continue

        # Skip TOC content
        if skip_toc:
            if re.match(r"^1\.\s", text):   # first real section
                skip_toc = False
            else:
                continue

        # Tag type
        if category == "Table":
            doc.metadata["type"] = "table"
        else:
            doc.metadata["type"] = "text"

        # Save cleaned content
        doc.page_content = text
        cleaned_docs.append(doc)

    print(f"Removed {removed} noisy chunks")
    print(f"Remaining chunks: {len(cleaned_docs)}")

    return cleaned_docs

In [7]:
docs_clean = clean_documents(docs)

Removed 21 noisy chunks
Remaining chunks: 460


In [8]:
# check if cleaning worked or not
print(docs[10])
print(docs_clean[10])

page_content='Tel: +1 (415) 700-9000 | hr@nexacorp.com | www.nexacorp.com' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'category_depth': 0, 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'last_modified': '2026-05-01T11:30:41', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'parent_id': '455472029b5887c8acdc6c539fd7b95a', 'category': 'UncategorizedText', 'element_id': '96a4b1466765e57ee816307d4291b752', 'type': 'text'}
page_content='' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'languages': ['eng'], 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'PageBreak', 'element_id': '374c9b677aede328ee9f4fdefb686eb5', 'type': 'text'}


Create Structured document